# CYCLOPS SENTINEL: Analisis Exploratorio de Datos - EDA

Analisis descriptivo y estadistico integral de los metadatos estructurales de correos para la deteccion de phishing. Este cuaderno examina distribuciones univariadas, valores atipicos, anomalias de longitud textual y correlaciones multivariadas de Pearson.

## Objetivos del Analisis
1. Analisis univariado: Distribucion de caracteres, conteo de palabras y puntaje de urgencia semantica.
2. Diagnostico de outliers: Evaluacion de dispersion con Rango Intercuartilico IQR y diagramas de caja.
3. Verificacion de balance de clases: Proporcion de correos legitimos frente a amenazas de phishing.
4. Relacion bivariada: Comportamiento de longitud textual frente al conteo de palabras mediante dispersion.
5. Matriz de correlacion multivariada: Mapa de calor de Pearson y validacion matematica de variables compuestas.


In [ ]:
# 1. Configuracion del Entorno y Librerias
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Segoe UI'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 120

os.makedirs('reports/figures', exist_ok=True)
print('[OK] Entorno de analisis exploratorio configurado con exito.')


## 2. Ingesta y Diagnostico Estructural del Dataset

Se carga el conjunto de datos limpio y unificado proveniente de repositorios de ciberseguridad. Se comprueban dimensiones, tipos de datos y ausencia total de valores nulos.


In [ ]:
# 2. Carga del Dataset Procesado
data_path = '../data/processed/clean_dataset.csv'
if not os.path.exists(data_path):
    data_path = r'C:/Users/Cristian Díaz/Documents/CYCLOPS/data/processed/clean_dataset.csv'

df = pd.read_csv(data_path)
print(f'Total de registros cargados: {len(df):,}')
print(f'Total de variables disponibles: {df.shape[1]}')
print(f'Valores nulos detectados: {df.isnull().sum().sum()}')
df.head(5)


## 3. Analisis Univariado: Longitud de Texto y Asimetria Positiva

Se evalua la distribucion de caracteres. En ciberseguridad, los correos benignos se concentran en longitudes breves, mientras que campanas maliciosas o volcados extensos generan una asimetria positiva severa con una cola larga hacia la derecha.


In [ ]:
# 3. Histograma y Estimacion de Densidad Kernel para Longitud de Caracteres
fig, ax = plt.subplots(figsize=(9, 4.5))

sns.histplot(
    data=df,
    x='char_count',
    kde=True,
    color='#0284c7',
    bins=40,
    ax=ax,
    line_kws={'linewidth': 2}
)

media_char = float(df['char_count'].mean())
mediana_char = float(df['char_count'].median())

ax.axvline(media_char, color='#ef4444', linestyle='--', linewidth=1.8, label=f'Media: {media_char:.1f}')
ax.axvline(mediana_char, color='#10b981', linestyle='-', linewidth=1.8, label=f'Mediana: {mediana_char:.1f}')

ax.set_title('Figura 1. Distribucion de Longitud de Texto: Asimetria Positiva Severa', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Numero de Caracteres - char_count', fontsize=10)
ax.set_ylabel('Frecuencia de Correos', fontsize=10)
ax.legend(frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.savefig('reports/figures/01_char_count_histogram.png', dpi=300)
plt.show()

print(f'Media aritmetica: {media_char:.2f} caracteres')
print(f'Mediana central:  {mediana_char:.2f} caracteres')
print(f'Coeficiente de asimetria: {df["char_count"].skew():.4f} - Sesgo marcado a la derecha')


## 4. Diagnostico de Outliers: Boxplot de Palabras y Rango Intercuartilico

El diagrama de caja ilustra la anatomia de cinco numeros: Minimo, Cuartil 1, Mediana, Cuartil 3 y Maximo. Los puntos aislados representan correos atipicos con densidad textual desproporcionada.


In [ ]:
# 4. Diagrama de Caja para Conteo de Palabras
fig, ax = plt.subplots(figsize=(9, 3))

sns.boxplot(
    data=df,
    x='word_count',
    color='#38bdf8',
    flierprops={'marker': 'o', 'markersize': 3, 'markerfacecolor': '#ef4444', 'alpha': 0.5},
    ax=ax
)

q1 = float(df['word_count'].quantile(0.25))
q3 = float(df['word_count'].quantile(0.75))
iqr = q3 - q1

ax.set_title('Figura 2. Dispersion del Conteo de Palabras y Diagnostico de Outliers', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Conteo de Palabras - word_count', fontsize=10)

plt.tight_layout()
plt.savefig('reports/figures/02_word_count_boxplot.png', dpi=300)
plt.show()

print(f'Primer Cuartil Q1: {q1:.1f} palabras')
print(f'Tercer Cuartil Q3: {q3:.1f} palabras')
print(f'Rango Intercuartilico IQR: {iqr:.1f} palabras')
print(f'Curtosis calculada: {df["word_count"].kurt():.4f} - Alta concentracion de valores extremos')


## 5. Balance de la Variable Objetivo: Correos Legitimos frente a Phishing

Un conjunto balanceado impide sesgos del modelo hacia una clase predominante. Se grafica la frecuencia discreta de la etiqueta `is_threat`.


In [ ]:
# 5. Grafico de Barras de Balance de Clases
fig, ax = plt.subplots(figsize=(6, 4))

class_counts = df['is_threat'].value_counts().sort_index()
labels = ['Legitimo - 0', 'Amenaza Phishing - 1']
colors = ['#10b981', '#ef4444']

bars = ax.bar(labels, class_counts.values, color=colors, width=0.5, edgecolor='#334155', linewidth=1.2)

for bar in bars:
    height = bar.get_height()
    pct = float(height) / float(len(df)) * 100.0
    ax.annotate(f'{height:,} - {pct:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4),
                textcoords='offset points',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, max(class_counts.values) * 1.15)
ax.set_title('Figura 3. Balance de la Variable Objetivo - is_threat', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Total de Correos', fontsize=10)

plt.tight_layout()
plt.savefig('reports/figures/03_target_balance_barplot.png', dpi=300)
plt.show()


## 6. Analisis Bivariado: Longitud de Caracteres frente a Conteo de Palabras

Se analiza la relacion bivariada entre caracteres y palabras. La fuerte alineacion diagonal confirma colinealidad antes de crear densidades compuestas.


In [ ]:
# 6. Diagrama de Dispersion Bivariado
fig, ax = plt.subplots(figsize=(8, 4.5))

sample_df = df.sample(n=min(5000, len(df)), random_state=42)
sns.scatterplot(
    data=sample_df,
    x='char_count',
    y='word_count',
    hue='is_threat',
    palette={0: '#10b981', 1: '#ef4444'},
    alpha=0.6,
    s=25,
    ax=ax
)

corr_val = float(df['char_count'].corr(df['word_count']))
ax.set_title(f'Figura 4. Relacion Bivariada: Caracteres vs Palabras - r = {corr_val:.2f}', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Conteo de Caracteres', fontsize=10)
ax.set_ylabel('Conteo de Palabras', fontsize=10)
ax.legend(title='Clase', labels=['Legitimo', 'Amenaza'], frameon=True, facecolor='white')

plt.tight_layout()
plt.savefig('reports/figures/04_bivariate_scatter.png', dpi=300)
plt.show()


## 7. Analisis Multivariado: Matriz de Calor de Pearson e Ingenieria de Variables

Se integra el mapa de calor multivariado de correlaciones de Pearson sobre las metricas numericas y las densidades compuestas url_density y urgency_density, verificando que aportan senales predictivas independientes.


In [ ]:
# 7. Mapa de Calor de Correlacion de Pearson y Metricas Compuestas
df['url_density'] = (df['url_count'] / np.maximum(df['word_count'], 1)) * 100
df['urgency_density'] = (df['urgency_score'] / np.maximum(df['word_count'], 1)) * 100
df['ip_exposure_score'] = df['has_ip'] * df['url_count']

corr_cols = [
    'char_count',
    'word_count',
    'url_count',
    'has_url',
    'has_ip',
    'urgency_score',
    'url_density',
    'urgency_density',
    'is_threat'
]

corr_matrix = df[corr_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(10, 7.5))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    linecolor='#1e293b',
    ax=ax,
    cbar_kws={'label': 'Coeficiente de Correlacion de Pearson'}
)

ax.set_title('Figura 5. Matriz de Correlacion de Pearson - Dataset CYCLOPS', fontsize=12, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('reports/figures/05_pearson_correlation_heatmap.png', dpi=300)
plt.show()

print('Correlaciones clave con la variable objetivo is_threat:')
for col in corr_cols[:-1]:
    r = float(df[col].corr(df['is_threat']))
    print(f' - {col:18s}: r = {r:+.4f}')


## 8. Tabla Ejecutiva de Estadisticas Descriptivas

Consolidacion de metricas descriptivas formales: Media, Desviacion Estandar, Mediana, IQR, Asimetria y Curtosis para cada variable analizada.


In [ ]:
# 8. Tabla Resumen Univariada y Bivariada
stats_df = pd.DataFrame({
    'Media': df[corr_cols].mean(),
    'Desv. Est.': df[corr_cols].std(),
    'Mediana': df[corr_cols].median(),
    'IQR': df[corr_cols].quantile(0.75) - df[corr_cols].quantile(0.25),
    'Asimetria': df[corr_cols].skew(),
    'Curtosis': df[corr_cols].kurt()
})

print('=== Tabla Descriptiva Formal de Metricas Estadisticas ===')
print(stats_df.round(4).to_string())
